[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239232-lesson-6-agent)

# Agent

## 回顧

我們建立了一個 router。

* 我們的 chat model 會根據使用者輸入，決定是否要發出 tool call
* 我們使用一條 conditional edge，路由到會呼叫 tool 的 node，或直接結束

![Screenshot 2024-08-21 at 12.44.33 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbac0ba0bd34b541c448cc_agent1.png)

## 目標

現在，我們可以把它擴充成一個通用的 agent 架構。

在上面的 router 中，我們呼叫了模型；如果它選擇呼叫 tool，我們就把一個 `ToolMessage` 回傳給使用者。

但是，如果我們把那個 `ToolMessage` *再傳回給模型*呢？

我們可以讓它選擇 (1) 呼叫另一個 tool，或是 (2) 直接回覆。

這正是 [ReAct](https://react-lm.github.io/) 這個通用 agent 架構背後的直覺。

* `act` —— 讓模型呼叫特定的 tool
* `observe` —— 把 tool 的輸出回傳給模型
* `reason` —— 讓模型針對 tool 的輸出進行推理，以決定下一步要做什麼（例如呼叫另一個 tool，或直接回覆）

這個[通用架構](https://blog.langchain.com/planning-for-agents/)可以套用到許多種類的 tool 上。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph langgraph-prebuilt

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

這裡，我們會用 [LangSmith](https://docs.langchain.com/langsmith/home) 來做 [tracing](https://docs.langchain.com/langsmith/observability-concepts)。

我們會把紀錄寫到一個名為 `langchain-academy` 的 project。

In [3]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

# 這會是一個 tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a and b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")

# 在這個 ipynb 中，我們把 parallel tool calling 設為 false，因為數學運算通常是循序進行的，而這次我們有 3 個可以做數學的 tool
# OpenAI 模型為了效率，特別會預設使用 parallel tool calling，參見 https://python.langchain.com/docs/how_to/tool_calling_parallel/
# 動手玩玩看，觀察模型在處理數學算式時的行為！
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)

讓我們建立 LLM，並用我們希望 agent 整體展現的行為來給它下 prompt。

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

如同之前，我們使用 `MessagesState`，並用我們的 tool 串列定義一個 `Tools` node。

`Assistant` node 就是我們綁定了 tool 的模型。

我們建立一個帶有 `Assistant` 與 `Tools` node 的 graph。

我們加入 `tools_condition` edge，它會根據 `Assistant` 是否呼叫了 tool，路由到 `End` 或 `Tools`。

現在，我們加入一個新步驟：

我們把 `Tools` node *接回* `Assistant`，形成一個迴圈。

* 在 `assistant` node 執行之後，`tools_condition` 會檢查模型的輸出是否為 tool call。
* 如果是 tool call，流程就會被導向 `tools` node。
* `tools` node 會接回 `assistant`。
* 只要模型持續決定要呼叫 tool，這個迴圈就會持續下去。
* 如果模型的回應不是 tool call，流程就會被導向 END，終止整個流程。

In [ ]:
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display

# Graph
builder = StateGraph(MessagesState)

# 定義 node：這些負責實際的工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定義 edge：這些決定控制流程如何移動
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果 assistant 最新的 message（結果）是 tool call -> tools_condition 會路由到 tools
    # 如果 assistant 最新的 message（結果）不是 tool call -> tools_condition 會路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# 顯示
display(Image(react_graph.get_graph(xray=True).draw_mermaid_png()))

In [11]:
messages = [HumanMessage(content="Add 3 and 4. Multiply the output by 2. Divide the output by 5")]
messages = react_graph.invoke({"messages": messages})

In [12]:
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Add 3 and 4. Multiply the output by 2. Divide the output by 5
================================== Ai Message ==================================
Tool Calls:
  add (call_i8zDfMTdvmIG34w4VBA3m93Z)
 Call ID: call_i8zDfMTdvmIG34w4VBA3m93Z
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: add

7
================================== Ai Message ==================================
Tool Calls:
  multiply (call_nE62D40lrGQC7b67nVOzqGYY)
 Call ID: call_nE62D40lrGQC7b67nVOzqGYY
  Args:
    a: 7
    b: 2
================================= Tool Message =================================
Name: multiply

14
================================== Ai Message ==================================
Tool Calls:
  divide (call_6Q9SjxD2VnYJqEBXFt7O1moe)
 Call ID: call_6Q9SjxD2VnYJqEBXFt7O1moe
  Args:
    a: 14
    b: 5
================================= Tool Message ===============

## LangSmith

我們可以在 LangSmith 中查看 trace。